# PANDA — Final submission notebook

**Settings:** GPU T4 ON, Internet OFF

**Attach on Kaggle:**
1. `prostate-cancer-grade-assessment` (official PANDA competition dataset)
2. Ordinal 5-fold weights dataset, containing `efficientnetb0_ordinal_fold{0..4}.pth`
3. `panda-src` (your private src code dataset, unless included in the weights dataset)
4. `efficientnetpytorch063` by optimo (offline package: https://www.kaggle.com/datasets/optimo/efficientnetpytorch063)

`PANDA: Resized Train Data (512x512)` is not a replacement for the official competition dataset; it does not provide the final submission test files.

In [ ]:
KAGGLE_INPUT = '/kaggle/input'
BATCH_SIZE = 16
ORDINAL_MODE = 'threshold'
N_FOLDS = 5
OUTPUT_CSV = '/kaggle/working/submission.csv'

# Current best confirmed family. Add extra 5-fold families here only after
# their OOF ensemble improves CV.
MODEL_FAMILIES = [
    {
        'name': 'b0_ordinal',
        'weights_slug': 'panda-effnetb0-ordinal-5fold',
        'weights_dir': None,
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
    },
]

In [ ]:
import glob
import importlib.util
import os
import sys

def find_existing_path(patterns, description, required=True):
    matches = []
    for pattern in patterns:
        if any(ch in pattern for ch in '*?['):
            matches.extend(glob.glob(pattern, recursive=True))
        elif os.path.exists(pattern):
            matches.append(pattern)
    matches = sorted(set(p for p in matches if os.path.exists(p)), key=lambda p: (len(p), p))
    if matches:
        return matches[0]
    if required:
        checked = '\n  - '.join(patterns)
        raise FileNotFoundError(f'Could not find {description}. Checked:\n  - {checked}')
    return None

def add_efficientnet_paths():
    added = []

    package_inits = glob.glob(os.path.join(KAGGLE_INPUT, '**', 'efficientnet_pytorch', '__init__.py'), recursive=True)
    for init_path in sorted(package_inits, key=lambda p: (len(p), p)):
        package_parent = os.path.dirname(os.path.dirname(init_path))
        if package_parent not in sys.path:
            sys.path.insert(0, package_parent)
            added.append(package_parent)

    wheel_paths = glob.glob(os.path.join(KAGGLE_INPUT, '**', '*efficientnet*.whl'), recursive=True)
    for wheel_path in sorted(wheel_paths, key=lambda p: (len(p), p)):
        if wheel_path not in sys.path:
            sys.path.insert(0, wheel_path)
            added.append(wheel_path)

    return added

efficientnet_paths = add_efficientnet_paths()
efficientnet_spec = importlib.util.find_spec('efficientnet_pytorch')
print('EfficientNet search paths:', efficientnet_paths or 'none found in attached inputs')
print('EfficientNet import location:', efficientnet_spec.origin if efficientnet_spec else 'not importable yet')

In [ ]:
import shutil
import numpy as np
import pandas as pd
import torch
import cv2
import skimage.io

def find_src_dir():
    inference_file = find_existing_path(
        [
            os.path.join(KAGGLE_INPUT, 'panda-src', 'src', 'inference.py'),
            os.path.join(KAGGLE_INPUT, 'panda-src', 'inference.py'),
            os.path.join(KAGGLE_INPUT, '**', 'src', 'inference.py'),
            os.path.join(KAGGLE_INPUT, '**', 'inference.py'),
        ],
        'panda src package',
    )
    src_dir = os.path.dirname(inference_file)
    expected = ['eval.py', 'model.py', 'dataset.py']
    missing = [name for name in expected if not os.path.exists(os.path.join(src_dir, name))]
    if missing:
        raise FileNotFoundError(f'Found {inference_file}, but src package is missing {missing}')
    return src_dir

working_src = '/kaggle/working/src'
src_dir = find_src_dir()
if os.path.exists(working_src):
    shutil.rmtree(working_src)
shutil.copytree(src_dir, working_src)
print('Copied src from:', src_dir)

def find_weights_dir(family):
    by_slug = find_existing_path(
        [
            os.path.join(KAGGLE_INPUT, family['weights_slug']),
            os.path.join(KAGGLE_INPUT, '**', family['weights_slug']),
        ],
        f"{family['name']} weights dataset",
        required=False,
    )
    if by_slug is not None:
        return by_slug

    first_weight = family['weight_pattern'].format(fold=0)
    weight_path = find_existing_path(
        [os.path.join(KAGGLE_INPUT, '**', first_weight)],
        f"{family['name']} fold-0 checkpoint",
        required=False,
    )
    if weight_path is None:
        print(f"WARN: {family['name']} weights were not found; inference will fall back to sample_submission if needed")
        return None
    return os.path.dirname(weight_path)

for family in MODEL_FAMILIES:
    family['weights_dir'] = find_weights_dir(family)
    print(f"{family['name']} weights:", family['weights_dir'])

sys.path.insert(0, '/kaggle/working')

try:
    import efficientnet_pytorch
except ModuleNotFoundError as exc:
    input_roots = sorted(glob.glob(os.path.join(KAGGLE_INPUT, '*')))
    raise ModuleNotFoundError(
        'efficientnet_pytorch is not available. Attach the optimo/efficientnetpytorch063 '
        'Kaggle dataset, or attach any offline dataset containing either an '
        'efficientnet_pytorch package folder or an efficientnet .whl file. '
        f'Current /kaggle/input roots: {input_roots}'
    ) from exc

from src.eval import round_preds
from src.inference import load_model, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Dataset that reads resized PNGs or official TIFF slide images directly
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGE_EXTS = ('.png', '.tiff', '.tif')

def slide_path(image_dir, image_id):
    for ext in IMAGE_EXTS:
        path = os.path.join(image_dir, f'{image_id}{ext}')
        if os.path.exists(path):
            return path
    return None

def coerce_rgb_image(img):
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.repeat(img[..., None], 3, axis=2)
    if img.shape[-1] == 4:
        img = img[..., :3]
    if img.shape[-1] != 3:
        raise ValueError(f'Expected an RGB image, got shape {tuple(img.shape)}')
    return np.ascontiguousarray(img)

def normalize_image(img):
    img = coerce_rgb_image(img)
    if np.issubdtype(img.dtype, np.integer):
        img = img.astype(np.float32) / np.iinfo(img.dtype).max
    else:
        img = img.astype(np.float32)
        if img.max() > 1.0:
            img = img / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return np.ascontiguousarray(img.transpose(2, 0, 1))

def read_slide_image(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.tiff', '.tif'):
        return skimage.io.MultiImage(path)[-1]
    return skimage.io.imread(path)

def count_matching_images(image_dir, image_ids):
    if image_dir is None or not os.path.isdir(image_dir):
        return 0
    return sum(slide_path(image_dir, image_id) is not None for image_id in image_ids)

def find_image_dir_for_ids(image_ids, preferred_dir=None):
    image_ids = [str(image_id) for image_id in image_ids]
    candidates = []
    for path in [
        preferred_dir,
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'train_images'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'train_images'),
    ]:
        if path is not None and os.path.isdir(path):
            candidates.append(path)

    candidates.extend(glob.glob(os.path.join(KAGGLE_INPUT, '**', 'test_images'), recursive=True))
    candidates.extend(glob.glob(os.path.join(KAGGLE_INPUT, '**', 'train_images'), recursive=True))
    candidates.extend(glob.glob(os.path.join(KAGGLE_INPUT, '**', 'train_images', 'train_images'), recursive=True))

    for image_id in image_ids[:5]:
        for ext in IMAGE_EXTS:
            for path in glob.glob(os.path.join(KAGGLE_INPUT, '**', f'{image_id}{ext}'), recursive=True):
                candidates.append(os.path.dirname(path))

    candidates = sorted(set(candidates), key=lambda p: (len(p), p))
    scored = [(count_matching_images(path, image_ids), path) for path in candidates]
    scored = sorted(scored, key=lambda item: (-item[0], len(item[1]), item[1]))
    if scored:
        best_count, best_path = scored[0]
        print(f'Best image dir: {best_path} ({best_count}/{len(image_ids)} matched)')
        if best_count == len(image_ids):
            return best_path
    else:
        print('Best image dir: None (0 matched)')
    return None

class SlideImageDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_dir):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        path = slide_path(self.image_dir, row.image_id)
        if path is None:
            raise FileNotFoundError(f'Could not find image for {row.image_id} in {self.image_dir}')
        img = read_slide_image(path)
        img = cv2.resize(img, (512, 512))
        img = normalize_image(img)
        return torch.from_numpy(img), torch.tensor(0.0)

In [ ]:
DATA = find_existing_path(
    [
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'test_images'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'test_images'),
    ],
    'PANDA test_images directory',
    required=False,
)
TEST = find_existing_path(
    [
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'test.csv'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'test.csv'),
    ],
    'PANDA test.csv',
    required=False,
)
SAMPLE = find_existing_path(
    [
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'sample_submission.csv'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'sample_submission.csv'),
        os.path.join(KAGGLE_INPUT, '**', 'sample_submission.csv'),
    ],
    'PANDA sample_submission.csv',
)
TRAIN = find_existing_path(
    [
        os.path.join(KAGGLE_INPUT, 'prostate-cancer-grade-assessment', 'train.csv'),
        os.path.join(KAGGLE_INPUT, 'competitions', 'prostate-cancer-grade-assessment', 'train.csv'),
        os.path.join(KAGGLE_INPUT, '**', 'train.csv'),
    ],
    'PANDA train.csv',
    required=False,
)

print('DATA:', DATA)
print('TEST:', TEST)
print('SAMPLE:', SAMPLE)
print('TRAIN:', TRAIN)

def prior_fallback_submission(df, train_csv):
    out = df[['image_id']].copy()
    default_grade = 2
    if train_csv is not None and os.path.exists(train_csv):
        train_df = pd.read_csv(train_csv)
        if 'isup_grade' in train_df.columns:
            default_grade = int(np.clip(round(train_df.isup_grade.mean()), 0, 5))
            if 'data_provider' in df.columns and 'data_provider' in train_df.columns:
                provider_prior = (
                    train_df.groupby('data_provider').isup_grade.mean()
                    .round().clip(0, 5).astype(int).to_dict()
                )
                out['isup_grade'] = (
                    df['data_provider'].map(provider_prior).fillna(default_grade).astype(int)
                )
                print('Using data_provider prior fallback:', provider_prior, 'default:', default_grade)
                return out
    out['isup_grade'] = default_grade
    print('Using global prior fallback:', default_grade)
    return out

test_df = None
try:
    sub_df = pd.read_csv(SAMPLE)

    if TEST is None:
        test_df = sub_df[['image_id']].copy()
    else:
        test_df = pd.read_csv(TEST)
        if 'image_id' not in test_df.columns:
            raise ValueError(f'test.csv missing image_id column: {test_df.columns.tolist()}')
    test_ids = test_df[['image_id']].copy()
    test_ids['isup_grade'] = 0

    DATA = find_image_dir_for_ids(test_ids.image_id.values, DATA)
    print('RESOLVED_DATA:', DATA)

    if os.path.exists(DATA or ''):
        test_dataset = SlideImageDataset(test_ids, DATA)
        print('Test slides:', len(test_dataset))

        all_family_preds = []
        for family in MODEL_FAMILIES:
            fold_preds = []
            for fold in range(N_FOLDS):
                weight_path = os.path.join(
                    family['weights_dir'],
                    family['weight_pattern'].format(fold=fold)
                )
                model = load_model(
                    weight_path,
                    backbone=family.get('backbone', 'efficientnet-b0'),
                    device=device,
                    model_kind=family.get('model_kind', 'baseline'),
                )
                preds = predict(
                    model, test_dataset, device,
                    batch_size=BATCH_SIZE,
                    ordinal_mode=ORDINAL_MODE,
                )
                fold_preds.append(preds)
                del model
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                print(f"{family['name']} fold {fold} done")
            all_family_preds.append(np.mean(fold_preds, axis=0))

        ensemble_preds = np.mean(all_family_preds, axis=0)
        final_preds = round_preds(ensemble_preds)
        sub_df = pd.DataFrame({'image_id': test_ids.image_id.values, 'isup_grade': final_preds})
    else:
        print('No matching image files found for test.csv rows; using metadata prior fallback')
        sub_df = prior_fallback_submission(test_df, TRAIN)
except Exception as e:
    import traceback
    print('Error:', e)
    traceback.print_exc()
    fallback_df = test_df if test_df is not None else pd.read_csv(SAMPLE)
    sub_df = prior_fallback_submission(fallback_df, TRAIN)

sub_df.to_csv('submission.csv', index=False)
print('Saved submission.csv')
print(sub_df.head())
